# Introduction to APIs

**Estimated time:** 15 minutes

Learn how an API request works, compare the resources in this module, and send
a request.


## Understand APIs for biomedical data

API stands for **application programming interface**. An API gives software a
standard way to request data or an action from another system.

We will use Python to request data for the paper's 25 gene symbols and examine
the responses. Unlike results copied from a website, API requests can be
repeated, checked, and shared.


### Identify the parts of an API request

First, identify the parts of a request.

| Term | Meaning |
|---|---|
| Endpoint | The web address used for a specific task |
| Request | The message sent to the API |
| Parameter | A value that controls the request, such as a gene or tissue ID |
| Response | The information returned by the API |
| JSON | A common text format used to organize returned data |

A reproducible request records its endpoint, parameters, resource version, and
date. Together, these details show exactly what was requested and when.


## Compare the APIs used in this module

GTEx, HuBMAP, and Pharos contain different biological data. Each API answers a
different question about the same 25 genes.

| CFDE resource | Information returned | Question this resource helps answer |
|---|---|---|
| [GTEx Portal API](https://gtexportal.org/api/v2/docs) | Median expression reported for named genes in selected heart tissues | Are these genes expressed in either heart tissue? |
| [HuBMAP APIs](https://docs.hubmapconsortium.org/apis.html) | Indexed expression availability and summaries for up to the first 500 ventricular cardiac-myocyte records | Which genes have indexed values in ventricular cardiac myocytes? |
| [Pharos GraphQL API](https://pharos.nih.gov/api) | Protein annotations, target development level, and selected knowledge counts | What is known about the encoded protein, and how developed is it as a target? |

The published variant table is saved in the repository. By default, GTEx,
HuBMAP, and Pharos are queried live. Dated responses are available as backups
when a service cannot be reached.

The prioritization lesson also uses the
[ProtVar API](https://www.ebi.ac.uk/ProtVar/api) for one missense VUS.
AlphaMissense and EVE predict whether the substitution may affect protein
function, while FoldX estimates its effect on protein stability. ProtVar is not
a CFDE resource.


## Build an E-utilities request

We will first use NCBI E-utilities because its endpoint and parameters are easy
to inspect in one URL. Build the URL by choosing a gene symbol, defining the
parameters, and joining them to the endpoint.

The completed example uses *MYH7*. Run the cell to display the NCBI request
URL without sending it.


In [ ]:
from urllib.parse import urlencode

# Choose a gene.
gene_symbol = "MYH7"

# Define the endpoint and search parameters.
endpoint = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
parameters = {
    "db": "gene",
    "term": f"{gene_symbol}[gene] AND Homo sapiens[organism]",
    "retmode": "json",
}

# Build the request URL.
request_url = f"{endpoint}?{urlencode(parameters)}"
request_url


The displayed URL sends `db=gene` to NCBI Gene, places the selected symbol and
`Homo sapiens` in `term`, and requests JSON with `retmode=json`. Change
`gene_symbol`, and the search changes with it. The endpoint and remaining
parameters stay the same.


### Send the request

Run the previous cell first. Then run this cell to send the request and
inspect selected fields from the JSON response.


In [ ]:
import requests

response = requests.get(request_url, timeout=30)
# Stop here if NCBI returns an error instead of interpreting an error page.
response.raise_for_status()

response_json = response.json()
search_result = response_json["esearchresult"]

# Display a small summary instead of the complete nested response.
{
    "status_code": response.status_code,
    "match_count": int(search_result["count"]),
    "ncbi_gene_ids": search_result["idlist"],
}


A status code of `200` confirms that NCBI received and processed the request.
The match count reports how many NCBI Gene records met the search terms, while
`ncbi_gene_ids` provides their database identifiers.


## Compare API request approaches

The three resources use two common request styles.


### Read a REST request

GTEx and HuBMAP provide REST-style APIs with endpoints for tasks such as
retrieving expression data or finding cells. Request parameters specify the
genes, tissues, or cells to return.


### Read a GraphQL query

**GraphQL** is an API query language. A GraphQL service often uses one endpoint.
The request contains a query that names the exact fields needed. Variables hold
values that can change, such as a gene symbol. The response follows the shape
of the query, so the client receives the fields it requested.

Pharos uses GraphQL. This query asks for a protein symbol, name, UniProt ID,
target development level, and publication count:

```graphql
# Give the request a name and declare the gene-symbol variable.
query TargetContext($symbol: String!) {
  # Request one target that matches the supplied symbol.
  target(q: {sym: $symbol}) {
    # Request the protein fields used in this module.
    sym
    name
    uniprot
    tdl
    publicationCount
  }
}
```

The fields in this query describe a protein target. The response follows the
same structure and contains only the requested fields.


## Quiz yourself!
Which line defines one of the NCBI request parameters?

- `"db": "gene"`
- `endpoint = "https://eutils.ncbi.nlm.nih.gov/..."`
- `response_json = response.json()`

<details>
<summary>Show answer and feedback</summary>

- **`"db": "gene"`:** Correct. The `db` parameter tells E-utilities to search the NCBI Gene database.
- **`endpoint = "https://eutils.ncbi.nlm.nih.gov/..."`:** This line defines the endpoint rather than a request parameter.
- **`response_json = response.json()`:** This line converts the returned JSON into a Python object after the request has been sent.

</details>

Which resource would you query to examine median gene expression in heart
tissue?

- GTEx
- HuBMAP
- Pharos

<details>
<summary>Show answer and feedback</summary>

- **GTEx:** Correct. GTEx reports median gene expression across human tissues, including the two heart tissues used in this module.
- **HuBMAP:** HuBMAP is used here to examine indexed expression in ventricular cardiac myocytes rather than bulk heart-tissue samples.
- **Pharos:** Pharos provides protein and target-development information rather than tissue expression.

</details>


## Key points

- An API request specifies an endpoint and the parameters that control the
  response.
- GTEx provides tissue expression, HuBMAP provides cell-type expression, and
  Pharos provides protein and target-development information.
- GTEx and HuBMAP use REST-style requests; Pharos uses GraphQL.
- ProtVar adds variant-level protein predictions for one selected missense VUS.
- Recording identifiers, versions, and retrieval dates makes a request easier
  to repeat.


## Other biomedical APIs

**Additional API examples:**
| API | Example use |
|---|---|
| [NCBI E-utilities](https://www.ncbi.nlm.nih.gov/home/develop/api/) | Search and retrieve records from NCBI resources such as Gene, PubMed, and Protein |
| [Ensembl REST API](https://rest.ensembl.org/) | Retrieve genes, variants, sequences, and comparative genomics data |
| [UCSC Genome Browser API](https://genome.ucsc.edu/goldenpath/help/api.html) | Retrieve genome sequences and selected annotation tracks |

**Next:** Inspect the 54 published variant rows and identify the 25 gene symbols
used in the API requests.
